# Análise Fatorial — Qualidade do Vinho (UCI)Dataset: Wine Quality (Red), 1.599 observações, 11 variáveis físico-químicas.

## Nível 1-2 (original): Preparação e testes de adequação iniciaisA análise original, com as 11 variáveis, indicou:- Bartlett: qui² = 8017,57, p < 0,001 (variáveis correlacionadas)- **KMO geral = 0,432 — inadequado** (abaixo do mínimo de 0,5)Isso significa que, apesar de existir correlação suficiente entre as variáveis (Bartlett), essa correlação está concentrada em poucas variáveis, não distribuída de forma homogênea o bastante para uma boa solução fatorial com todas as 11 variáveis.

## Nível 2.5 (refinamento): Remoção iterativa de variáveis com baixo KMO individualPrática padrão em análise fatorial: quando o KMO geral é inadequado, remove-se a variável com **menor KMO individual (MSA)**, recalcula-se o KMO geral, e repete-se até a adequação melhorar.

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.preprocessing import StandardScalerfrom factor_analyzer import calculate_kmocols_names = ['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',              'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','alcohol','quality']df = pd.read_csv('winequality-red.csv', header=None, names=cols_names)X = df.drop(columns=['quality'])Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)def run_kmo(data):    kmo_all, kmo_model = calculate_kmo(data)    return kmo_model, pd.Series(kmo_all, index=data.columns).sort_values()cols = list(Xs.columns)step = 1while len(cols) > 3:    kmo_model, kmo_series = run_kmo(Xs[cols])    print(f"Passo {step}: n_vars={len(cols)}  KMO_geral={kmo_model:.3f}  pior_var='{kmo_series.index[0]}' ({kmo_series.iloc[0]:.3f})")    if kmo_model >= 0.6 and kmo_series.iloc[0] >= 0.5:        print(">>> KMO geral adequado e todas as variáveis com MSA >= 0.5, parar remoção")        break    cols.remove(kmo_series.index[0])    step += 1print("\nVariáveis finais:", cols)

Passo 1: n_vars=11  KMO_geral=0.432  pior_var='residual sugar' (0.205)Passo 2: n_vars=10  KMO_geral=0.499  pior_var='alcohol' (0.330)Passo 3: n_vars=9  KMO_geral=0.536  pior_var='chlorides' (0.359)Passo 4: n_vars=8  KMO_geral=0.589  pior_var='total sulfur dioxide' (0.390)Passo 5: n_vars=7  KMO_geral=0.670  pior_var='free sulfur dioxide' (0.356)Passo 6: n_vars=6  KMO_geral=0.680  pior_var='sulphates' (0.770)>>> KMO geral adequado e todas as variáveis com MSA >= 0.5, parar remoçãoVariáveis finais: ['fixed acidity', 'volatile acidity', 'citric acid', 'density', 'pH', 'sulphates']

## Nível 3 (refeito): Determinação do número de fatores — solução final

In [ ]:
from factor_analyzer import FactorAnalyzer, calculate_bartlett_sphericityfinal_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'density', 'pH', 'sulphates']Xf = Xs[final_cols]chi2, p = calculate_bartlett_sphericity(Xf)kmo_all, kmo_model = calculate_kmo(Xf)print(f"Bartlett: qui2={chi2:.2f}, p={p:.2e}")print(f"KMO geral: {kmo_model:.3f} (adequado)")print(pd.Series(kmo_all, index=final_cols).round(3).sort_values(ascending=False))fa = FactorAnalyzer(rotation=None, n_factors=len(final_cols))fa.fit(Xf)ev, v = fa.get_eigenvalues()print("\nAutovalores:", np.round(ev, 3))print("Fatores por Kaiser (>1):", int(sum(ev > 1)))

Bartlett: qui2=3974.04, p=0.00e+00KMO geral: 0.680 (adequado)sulphates           0.770citric acid         0.748pH                  0.744fixed acidity       0.644volatile acidity    0.613density             0.599dtype: float64Autovalores: [2.936 1.202 0.824 0.575 0.296 0.167]Fatores por Kaiser (>1): 2

## Nível 4 (refeito): Extração dos fatores (rotação varimax)

In [ ]:
fa2 = FactorAnalyzer(rotation='varimax', n_factors=2)fa2.fit(Xf)var, prop, cum = fa2.get_factor_variance()print("Variância explicada por fator (%):", np.round(prop*100, 2))print("Variância acumulada (%):", np.round(cum*100, 2))loadings = pd.DataFrame(fa2.loadings_, index=final_cols, columns=['Fator1', 'Fator2'])print("\nCargas fatoriais (varimax):")print(loadings.round(3))commu = fa2.get_communalities()print("\nComunalidades:")print(pd.Series(commu, index=final_cols).round(3).sort_values())

Variância explicada por fator (%): [32.99 24.05]Variância acumulada (%): [57.03]Cargas fatoriais (varimax):                  Fator1  Fator2fixed acidity      0.956   0.302volatile acidity   0.014  -0.792citric acid        0.502   0.709density            0.680   0.009pH                -0.576  -0.338sulphates          0.138   0.327Comunalidades:sulphates           0.126pH                  0.447density             0.462volatile acidity    0.628citric acid         0.754fixed acidity       1.005

## Interpretação final**Fator 1 (32,99% da variância) — "Corpo/Acidez fixa":** cargas altas em `fixed acidity` (0,956), `density` (0,680) e, com carga negativa, `pH` (-0,576). Representa o eixo de acidez estrutural e corpo do vinho — vinhos mais ácidos tendem a ter maior densidade e menor pH.**Fator 2 (24,05% da variância) — "Acidez volátil vs. frescor":** oposição entre `volatile acidity` (-0,792) e `citric acid` (0,709). Representa a tensão entre acidez volátil (associada a defeitos/deterioração) e ácido cítrico (associado a frescor e qualidade sensorial).Juntos, os 2 fatores explicam **57,03%** da variância — abaixo dos 70% de referência, mas consistente com uma solução estatisticamente adequada (KMO = 0,680), diferente da solução inicial com 11 variáveis, que tinha maior variância explicada nominal (70,8%) mas era construída sobre uma base estatisticamente inadequada (KMO = 0,432).`sulphates` tem comunalidade baixa (0,126) mesmo com KMO individual alto — participa da estrutura de correlação geral, mas não se associa fortemente a nenhum dos 2 fatores extraídos.